In [22]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

True

In [23]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


In [24]:
query_com = """
SELECT DISTINCT v.Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2026-07-25'
    AND v.Fecha <= '2026-07-31'
    AND Linea = 'VERBENA'
    AND v.Venta_Neta > 0
GROUP BY  v.Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine)

df_ventas 

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,101445,C63323877,LILIA AIDEE VELASCO,lvelasco2509@hotmail.com,3168345938,2026-07-25,Shopify,211755.60
1,101452,C13816945,HORACIO MARTINEZ,curiacio@yahoo.com,3128510678,2026-07-25,Shopify,106718.10
2,101456,C1030600884,YULY RUIZ,yulyruiz91@hotmail.com,3107806468,2026-07-25,Shopify,345363.30
3,101477,C1128283457,KATERIN GOEZ,kategoez.c@gmail.com,3218522615,2026-07-26,Shopify,202512.30
4,101481,C1136883772,PAOLA ANDREA NAJAR GOMEZ,andreanajar3@hotmail.com,3014307520,2026-07-27,Shopify,150413.70
...,...,...,...,...,...,...,...,...
436,4354481672,C2000020427,FRANCISCA MEZA,fmeza2002@gmail.com,3163188762,2026-07-31,Retail,112815.13
437,4354481691,C46387220,GUIOMAR ADRIANA VARGAS TAMAYO,adrianavargastamayo@gmail.com,1111111111,2026-07-31,Retail,226890.76
438,4354481721,C8486566,JORGE ENRIQUE GONZALEZ ALMENDRALES,jegait83@gmail.com,3174351555,2026-07-31,Retail,112815.13
439,4354481778,C523335,JESSICA HUA ZHENG MORA,jessicahua0812@gmail.com,3113338578,2026-07-31,Retail,225630.26


In [25]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Pedido,
# 	v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta,
#     Canal,
#     SUM(Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'JULIO_2026')
# 	AND Fecha BETWEEN '2026-07-04' AND '2026-07-31'
# GROUP BY  v.Pedido,
# 	v.Cliente,
#     cc.Nombres + ' ' +  cc.Apellidos,
#     cc.Email,
#     cc.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [26]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['email', 'campaignName', 'sentAt', 'openedAt', 'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'email': 'Email',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'openedAt': 'apertura',
    'clicks': 'clicks'
}, inplace=True)

# 1. Convertir la columna openedAt de string a lista real de fechas
df_campania['apertura'] = df_campania['apertura'].astype(str)
df_campania['apertura'] = df_campania['apertura'].apply(
    lambda x: ast.literal_eval(x) if x.startswith("[") else []
)

# 2. Expandir: cada fecha en una fila
df_expanded = df_campania.explode('apertura').copy()

# 3. Limpiar y convertir a datetime
df_expanded['apertura'] = df_expanded['apertura'].str.strip()
df_expanded['apertura'] = pd.to_datetime(
    df_expanded['apertura'],
    format='%Y-%m-%dT%H:%M:%S.%fZ',
    utc=True,
    errors='coerce'
).dt.date

#Convertir la columna clicks en Fecha
df_campania['clicks'] = df_campania['clicks'].astype(str)

def extraer_fecha_click(valor):
    try:
        # Convertir string a objeto Python (lista de dicts)
        lista = ast.literal_eval(valor)
        if isinstance(lista, list) and len(lista) > 0 and 'clicked_at' in lista[0]:
            fecha = pd.to_datetime(lista[0]['clicked_at'], utc=True, errors='coerce')
            return fecha.date()  # solo fecha
    except Exception:
        return pd.NaT
    return pd.NaT

df_campania['clicks'] = df_campania['clicks'].apply(extraer_fecha_click)

# 4. Normalizar correos
df_expanded['Email'] = df_expanded['Email'].str.lower().str.strip()

# 5. Elegir solo la primera apertura por email
df_abiertos = df_expanded.dropna(subset=['apertura']) \
    .sort_values('apertura') \
    .drop_duplicates(subset='Email', keep='first')

df_clics =  df_campania[df_campania['clicks'].notnull()].copy()

# Filtro correos abiertos hasta una fecha maxima
fecha_maxima = pd.to_datetime('2026-07-31').date()
df_abiertos_filtrados = df_abiertos[df_abiertos['apertura'] <= fecha_maxima]
df_clicks_filtrados = df_clics[df_clics['clicks'] <= fecha_maxima]
total_abiertos = df_abiertos_filtrados['Email'].nunique()
total_clicks = df_clicks_filtrados['Email'].nunique()

In [27]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_abiertos, on='Email', how='inner')

# Filtrar ventas dentro de 5 días desde la apertura
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['apertura']).apply(lambda x: x.days)
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 5)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Número de clientes que abrieron hasta {fecha_maxima}: {total_abiertos}")
print(f"Número de clientes que hicieron clics hasta {fecha_maxima}: {total_clicks}")
print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.2f}"  )
# Resultado final
df_atribucion.head()

Número de clientes que abrieron hasta 2026-07-31: 3366
Número de clientes que hicieron clics hasta 2026-07-31: 97
Total de Ventas: 25
Total de venta atribuida: 3,531,945.50


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,fecha_envio,apertura,clicks,dias_post_apertura
0,101445,C63323877,LILIA AIDEE VELASCO,lvelasco2509@hotmail.com,3168345938,2026-07-25,Shopify,211755.60,VERBENA CÃTRICA,2026-07-25T15:00:55.000Z,2026-07-25,"[{'clicked_at': '2026-07-25T15:12:28.000Z', 'c...",0
1,101481,C1136883772,PAOLA ANDREA NAJAR GOMEZ,andreanajar3@hotmail.com,3014307520,2026-07-27,Shopify,150413.70,VERBENA CÃTRICA,NaN,2026-07-26,[],1
2,101522,C79455183,WILLIAM HERNAN VASQUEZ TORRES,whvasqueztorres@gmail.com,3157614633,2026-07-29,Shopify,40334.40,VERBENA CÃTRICA,2026-07-25T15:00:49.000Z,2026-07-25,[],4
3,101535,C10016676,ANDRES FELIPE ZULUAGA MONTOYA,felipezuluaga73@hotmail.com,3216413690,2026-07-29,Shopify,127851.64,VERBENA CÃTRICA,2026-07-25T15:00:45.000Z,2026-07-25,"[{'clicked_at': '2026-07-25T15:10:57.000Z', 'c...",4
4,101576,C35467207,MARIA HERNANDEZ,mariehdez@hotmail.com,3114821492,2026-07-30,Shopify,80038.57,VERBENA CÃTRICA,2026-07-25T15:00:54.000Z,2026-07-25,[],5


In [28]:
df_atribucion.to_excel('Ventas.xlsx', index=False)